<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-08-tool-use-and-mcp/lesson-8.1-function-calling-for-mcp/notebooks/exercises-8.1.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 8.1 — Function Calling for MCP
Netsetos GenAI Course v2.0 | Module 8: AI Agents

Three-provider comparison, tool choice modes, token economics, error handling, security, production patterns, India stack


In [ ]:
print('Module 8: AI Agents & MCP')
print('Lesson 8.1: Function Calling — The Foundation')


## Ex 1: Tool Choice Modes


In [ ]:
print('Tool choice modes across providers:')
print()
for mode, openai, gemini, claude in [
    ('Model decides','"auto"','AUTO','auto'),
    ('No tools','"none"','NONE','none'),
    ('Must call','"required"','ANY','any'),
    ('Force specific','{"type":"function","name":"X"}','ANY+allowed_names','{"type":"tool","name":"X"}'),
]: print(f'  {mode:18s}: {openai:35s} | {gemini:20s} | {claude}')
print()
print('Parallel calling:')
print('  OpenAI: default on (GPT-4o/4.1/5), NOT supported on o-series')
print('  Gemini: multi-part response with unique IDs')
print('  Claude: ~100% success on Claude 4, disable with disable_parallel_tool_use=true')


## Ex 2: Token Cost Calculator


In [ ]:
def tool_token_cost(num_tools, tokens_per_tool=125, turns=10, price_per_mtok=2.50):
    per_request = num_tools * tokens_per_tool
    per_conversation = per_request * turns
    cost_per_1000_convos = (per_conversation * 1000 / 1_000_000) * price_per_mtok
    return per_request, per_conversation, cost_per_1000_convos

print('Token cost of tool definitions (GPT-4o @ $2.50/MTok input):')
print(f'{"Tools":>6s} {"Per Request":>12s} {"Per 10-Turn":>12s} {"Per 1K Convos":>14s}')
for n in [5, 10, 20, 50, 100]:
    pr, pc, cost = tool_token_cost(n)
    print(f'{n:>6d} {pr:>10,d} tok {pc:>10,d} tok ${cost:>12.2f}')
print()
print('With prompt caching (90% discount):')
for n in [20, 50, 100]:
    _, _, cost_full = tool_token_cost(n)
    _, _, cost_cached = tool_token_cost(n, price_per_mtok=0.25)
    print(f'  {n} tools: ${cost_full:.2f} → ${cost_cached:.2f} (save ${cost_full-cost_cached:.2f})')


## Ex 3: Strict Mode + Pydantic Validation


In [ ]:
print('Two-layer defense for production function calling:')
print()
print('Layer 1: Strict mode (prevents malformed JSON at generation)')
print('  OpenAI: strict: True in tool definition')
print('  Claude: strict: True in tool definition')
print('  Requirements: all properties in required, additionalProperties: false')
print()
print('Layer 2: Pydantic validation (validates before execution)')
print('  from pydantic import BaseModel, Field, field_validator')
print('  class SearchArgs(BaseModel):')
print('      keywords: list[str] = Field(min_length=1, max_length=10)')
print('      category: str = Field(pattern="^(electronics|clothing|books)$")')
print()
print('Error handling:')
print('  1. Parse JSON → catch JSONDecodeError')
print('  2. Validate with Pydantic → catch ValidationError')
print('  3. Return error TO LLM for self-correction (max 3 attempts)')
print('  4. NEVER silently fallback to empty dict!')


## Ex 4: Security Guardrails


In [ ]:
print('Security guardrails for tool-calling systems:')
print()
for threat, defense in [
    ('Prompt injection via tool results','Treat LLM as untrusted intermediary'),
    ('Unauthorized tool access','Role-based tool filtering BEFORE sending to LLM'),
    ('SQL/command injection in args','Input sanitization + length limits'),
    ('Excessive permissions','Least privilege: user permissions, not system'),
    ('High-risk operations','Human-in-the-loop confirmation gate'),
    ('Silent failures','Comprehensive audit logging (name, args, output, user, cost)'),
]: print(f'  Threat: {threat}')
   print(f'  Defense: {defense}')
   print()


## Ex 5: Dynamic Tool Loading


In [ ]:
print('Dynamic Context Loading (DCL) — 3 levels:')
print()
for level, what, tokens in [
    ('L1: Categories','10 category descriptions','~500 tokens'),
    ('L2: Summaries','20 tool summaries for selected category','~2,000 tokens'),
    ('L3: Full schemas','3 full schemas for selected tools','~450 tokens'),
]: print(f'  {level:20s}: {what:45s} | {tokens}')
print(f'  {"Total":20s}: {"~3,000 tokens":45s} | vs 30,000+ for all tools')
print()
print('Automated solutions:')
print('  OpenAI: tool search (GPT-5.4+, defer_loading: true)')
print('  Anthropic: Tool Search Tool (85% token reduction)')
print('  Manual: "unlock" meta-tool pattern')


## Ex 6: Function Calling → MCP → Agents


In [ ]:
print('Evolution: Function Calling → MCP → Agents')
print()
for stage, year, description in [
    ('Function calling','2023','Vendor-specific, single-turn, stateless'),
    ('Tool use','2024','Multi-turn, parallel calls, provider-trained tools'),
    ('MCP','2024-25','JSON-RPC 2.0, N×M → M+N, standardized protocol'),
    ('Agents','2025-26','Tool search, programmatic calling, multi-agent teams'),
]: print(f'  {stage:20s} ({year}): {description}')
print()
print('Key insight: these layers COEXIST in production')
print('  Function calling = reliable foundation (atomic primitive)')
print('  MCP = standardized integration protocol')
print('  Agent frameworks = orchestration layer')
print('  ReAct loop: reason → ACT (= function call) → observe → loop')


## Ex 7: Provider Pricing Comparison


In [ ]:
print('Function calling pricing (per 1M tokens, April 2026):')
print()
for model, input_p, output_p, best_for in [
    ('GPT-4.1','$2.00','$8.00','Best price-performance for tools'),
    ('GPT-4o','$2.50','$10.00','Multimodal + tools'),
    ('GPT-4o-mini','$0.15','$0.60','High-volume, simple tools'),
    ('Gemini 2.5 Flash','$0.30','$2.50','Cheapest tool calling'),
    ('Gemini 2.5 Pro','$2.50','$15.00','Complex reasoning + tools'),
    ('Claude Haiku 4.5','$1.00','$5.00','Fast, affordable tools'),
    ('Claude Sonnet 4.6','$3.00','$15.00','Best quality tools'),
    ('Sarvam-30b','INR pricing','INR pricing','Indic languages (2-6× cheaper)'),
]: print(f'  {model:20s}: {input_p:>8s}/{output_p:>8s}  | {best_for}')


## Ex 8: India Stack


In [ ]:
print('India function calling stack:')
print()
print('1. Sarvam AI (OpenAI-compatible):')
print('   base_url="https://api.sarvam.ai/v1"')
print('   Models: sarvam-30b (recommended), sarvam-105b')
print('   Advantage: Indic tokenizer ~1:1 ratio (GPT-4: 2-6× more tokens)')
print()
print('2. Bhashini (free APIs):')
print('   ASR, Translation, TTS, OCR, Transliteration')
print('   22 scheduled Indian languages')
print('   Wrap as function calling tools')
print()
print('3. Translate-first pattern:')
print('   Detect language → Translate to English →')
print('   Function calling (English, accurate) →')
print('   Translate results back → Respond in user language')
print('   Fixes: 39% → 60%+ accuracy for Indic function calling')


---
## Recap
Function calling is the atomic primitive powering MCP and agents. Tool definitions consume ~100-150 tokens each — use dynamic loading and prompt caching (90% discount). Strict mode + Pydantic validation = production reliability. Never silently fallback on parse errors. Security: role-based tool access, input sanitization, audit logging. Three providers converge: 128 tool limits, parallel calling, strict schemas, tool search. The evolution: function calling → tool use → MCP → agents — all layers coexist. For India: Sarvam's OpenAI-compatible API with efficient Indic tokenizer + Bhashini free language tools + translate-first pattern for reliable Indic tool routing.
